# Transformer-CTC — Execution pas à pas (Notebook)

Objectif: exécuter le script `transformerstemp.py` **fonction par fonction** dans l'ordre logique, avec explications.

## Etape 0 — Configuration (chemins + hyperparams + modèle)
Cette étape regroupe **toute la configuration** (données, hyperparams, modèle). Elle correspond aux blocs de config en haut du script.

In [6]:
# Imports (même ordre que le script)
import os
import json
import time
import shutil
import math
from datetime import datetime
from typing import List, Dict, Optional
import csv

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.backends.cudnn as cudnn
import h5py
from scipy.ndimage import gaussian_filter1d

In [7]:
def set_seed(seed: int):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


In [8]:
# Chemins de donnees (versions locales reelles)
import os
from pathlib import Path



PROJECT_ROOT = Path.cwd()

RUNS_ROOT = str(PROJECT_ROOT / "outputs" / "runs")
DATA_ROOT = str(PROJECT_ROOT / "data")
SPLIT_CSV = str(PROJECT_ROOT / "data" / "t15_sessions_random_split.csv")

os.makedirs(RUNS_ROOT, exist_ok=True)


In [9]:
# Clés HDF5
NEURAL_KEY = "input_features"   # [T,512] float32
LABELS_KEY = "seq_class_ids"    # [L] int64 (padding 0 a droite)

In [10]:
# Hyperparams modele
N_FEATURES = 512
N_CLASSES = 41
BLANK_ID = 0

TF_D_MODEL = 256
TF_LAYERS = 12
TF_FF_DIM = 1024
TF_NHEAD = 8
TF_DROPOUT = 0.1

PATCH_SIZE = 14
PATCH_STRIDE = 4

DAY_INPUT_DROPOUT = 0.2


In [11]:
# Hyperparams entrainement
BASE_LR = 2e-4          # taux d'apprentissage initial (AdamW)
WEIGHT_DECAY = 1e-5     # regularisation L2 (penalise les poids)
CLIP_NORM = 1.0         # norme max pour le gradient clipping
EPOCHS = 50            # nombre total d'epochs
PRINT_EVERY = 50        # frequence d'affichage des logs (batches)
USE_AMP = True          # active l'entrainement en precision mixte
SEED = 1337             # seed de reproductibilite
PRELOAD_RAM = False     # charge tout le train en RAM au demarrage
PRELOAD_VAL = False     # charge tout le val en RAM au demarrage

BATCH_SIZE = 8          # taille de batch (nb de trials par batch)
AMP_WARMUP_EPOCHS = 2   # epochs sans AMP pour stabiliser au debut
LR_WARMUP_EPOCHS = 3    # warmup LR (gardee en config, non utilisee ici)
EVAL_EVERY = 2          # evaluation toutes les N epochs
SAVE_LAST_EVERY = 5     # sauvegarde du dernier modele toutes les N epochs
EMPTY_CACHE_EVERY = 10  # libere cache GPU toutes les N epochs

AUG_SMOOTH_ENABLE = True       # active le lissage gaussien en data-aug
AUG_SMOOTH_PROB = 0.5          # probabilite d'appliquer le lissage
AUG_SMOOTH_STD = 2.0           # ecart-type du noyau gaussien
AUG_SMOOTH_SIZE = 100          # taille du noyau de lissage
AUG_SMOOTH_PADDING = "same"   # type de padding (same/valid)
AUG_SMOOTH_EVAL_ENABLE = True  # applique aussi le lissage en eval

RANDOM_CUT_MAX = 5     # coupe aleatoire max au debut des sequences
NOISE_STD = 0.5        # ecart-type du bruit blanc additif
OFFSET_STD = 0.1       # ecart-type de l'offset constant par trial

ACCUM_STEPS = 4        # accumulation: on additionne 4 mini-batchs avant 1 update (gradients accumules puis step)
WARMUP_FRAC = 0.05     # fraction du total des steps pour le warmup


## Etape 1 ? Lecture du CSV (split sessions)
On lit `t15_sessions_random_split.csv` pour recuperer les dates `type == train`.
Ces dates sont ensuite converties en noms de dossiers `t15.YYYY.MM.DD`.

In [12]:
def load_train_eval_dates_from_csv(split_csv: str):
    """
    Lit le CSV (date, type) et renvoie deux listes identiques:
      - train_dates : ['t15.2023.08.11', ...]
      - eval_dates  : ['t15.2023.08.11', ...]
    On ne garde que les lignes avec type == 'train'.
    """
    train_dates = []
    eval_dates = []

    with open(split_csv, "r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            date = row["date"].strip()          # ex: '2023-08-11'
            typ = row["type"].strip().lower()   # 'train', 'eval', etc.

            if typ != "train":
                continue
            if not date:
                continue

            folder_name = "t15." + date.replace("-", ".")  # 't15.2023.08.11'

            # meme liste pour train et eval
            train_dates.append(folder_name)
            eval_dates.append(folder_name)

    return train_dates, eval_dates

TRAIN_DATES, EVAL_DATES = load_train_eval_dates_from_csv(SPLIT_CSV)
print(f"[CSV] TRAIN={len(TRAIN_DATES)} dates | EVAL={len(EVAL_DATES)} dates")
print(TRAIN_DATES[:5])


[CSV] TRAIN=7 dates | EVAL=7 dates
['t15.2023.08.13', 't15.2023.08.18', 't15.2023.08.20', 't15.2023.08.25', 't15.2023.08.27']


In [13]:
# Sessions -> day index (necessaire pour DayInputLayer)
SESSIONS = TRAIN_DATES
SESSION_TO_ID = {}
for i, name in enumerate(SESSIONS):
    SESSION_TO_ID[name] = i
N_DAYS = len(SESSIONS)
print('N_DAYS:', N_DAYS)


N_DAYS: 7


**Exemple de sortie** (avec le CSV actuel):
```
[CSV] TRAIN=... dates | EVAL=... dates
['t15.2023.08.13', 't15.2023.08.18', 't15.2023.08.20', ...]
```

## Etape 2 ? Construction des chemins HDF5
On transforme chaque date `t15.YYYY.MM.DD` en chemin vers `data_train.hdf5` et `data_val.hdf5`.
Les fichiers absents sont ignores avec un warning.

In [14]:
def build_h5_paths_from_dates(root: str, date_folders, filename: str = "data_train.hdf5"):
    """
    A partir d'une liste de dossiers de dates (ex: 't15.2023.08.11'),
    construit la liste des chemins complets vers filename
    (par defaut data_train.hdf5).
    """
    paths = []
    for folder in date_folders:
        full_path = os.path.join(root, folder, filename)
        if not os.path.isfile(full_path):
            print(f"[WARN] Fichier absent : {full_path}")
            continue
        paths.append(full_path)
    return paths

# Train : fichiers data_train.hdf5
H5_PATHS = build_h5_paths_from_dates(DATA_ROOT, TRAIN_DATES, filename="data_train.hdf5")
# Val : fichiers data_val.hdf5
EVAL_H5_PATHS = build_h5_paths_from_dates(DATA_ROOT, EVAL_DATES, filename="data_val.hdf5")

print(f"[HDF5] TRAIN files={len(H5_PATHS)} | VAL files={len(EVAL_H5_PATHS)}")
print("[HDF5] TRAIN first 3:\n", H5_PATHS[:3])
print("[HDF5] VAL first 3:\n", EVAL_H5_PATHS[:3])


[HDF5] TRAIN files=7 | VAL files=7
[HDF5] TRAIN first 3:
 ['c:\\code\\brain2text_bundle\\brain2text_project\\data\\t15.2023.08.13\\data_train.hdf5', 'c:\\code\\brain2text_bundle\\brain2text_project\\data\\t15.2023.08.18\\data_train.hdf5', 'c:\\code\\brain2text_bundle\\brain2text_project\\data\\t15.2023.08.20\\data_train.hdf5']
[HDF5] VAL first 3:
 ['c:\\code\\brain2text_bundle\\brain2text_project\\data\\t15.2023.08.13\\data_val.hdf5', 'c:\\code\\brain2text_bundle\\brain2text_project\\data\\t15.2023.08.18\\data_val.hdf5', 'c:\\code\\brain2text_bundle\\brain2text_project\\data\\t15.2023.08.20\\data_val.hdf5']


## Etape 3 ? Lister les trials dans chaque HDF5
On recupere toutes les cles `trial_###` pour chaque fichier HDF5.

In [15]:
def _list_trial_keys(file_path: str):
    with h5py.File(file_path, "r") as f:
        keys = []
        for k in f.keys():
            if k.startswith("trial_") and k[6:].isdigit():
                keys.append(k)
    keys.sort()
    return keys

# Exemple: lister les trials du 1er fichier train (si existe)
if len(H5_PATHS) > 0:
    example_keys = _list_trial_keys(H5_PATHS[0])
    print(f"[TRIALS] {H5_PATHS[0]} -> {len(example_keys)} trials")
    print(example_keys[:5])


[TRIALS] c:\code\brain2text_bundle\brain2text_project\data\t15.2023.08.13\data_train.hdf5 -> 348 trials
['trial_0000', 'trial_0001', 'trial_0002', 'trial_0003', 'trial_0004']


## Etape 4 ? Construire les index train/val
On cree des listes d'objets (dict) qui pointent vers chaque trial : fichier, cle, numero, etc.

In [16]:
def build_train_index_from_hdf5():
    idx = []
    gid = 0
    for file_path in H5_PATHS:
        keys = _list_trial_keys(file_path)
        for k in keys:
            idx.append({
                "global_id": gid,
                "file_path": file_path,
                "trial_key": k,
                "trial_num": int(k[6:]),
            })
            gid += 1
    print(f"[TRAIN] Trials total indexes: {len(idx)}")
    return idx


def build_eval_index_from_hdf5():
    val_idx = []
    gid = 0
    for file_path in EVAL_H5_PATHS:
        day_folder = os.path.basename(os.path.dirname(file_path))
        day_idx = SESSION_TO_ID[day_folder]
        keys = _list_trial_keys(file_path)
        for k in keys:
            val_idx.append({
                "day_idx": day_idx,
                "global_id": gid,
                "file_path": file_path,
                "trial_key": k,
                "trial_num": int(k[6:]),
            })
            gid += 1
        print(f"[EVAL] {os.path.basename(file_path)} -> {len(keys)} trials")
    print(f"[EVAL] Trials total (tous fichiers): {len(val_idx)}")
    return val_idx


train_index = build_train_index_from_hdf5()
val_index = build_eval_index_from_hdf5()
print(train_index[:2])
print(val_index[:2])


[TRAIN] Trials total indexes: 1680
[EVAL] data_val.hdf5 -> 35 trials
[EVAL] data_val.hdf5 -> 49 trials
[EVAL] data_val.hdf5 -> 48 trials
[EVAL] data_val.hdf5 -> 25 trials
[EVAL] data_val.hdf5 -> 25 trials
[EVAL] data_val.hdf5 -> 49 trials
[EVAL] data_val.hdf5 -> 34 trials
[EVAL] Trials total (tous fichiers): 265
[{'global_id': 0, 'file_path': 'c:\\code\\brain2text_bundle\\brain2text_project\\data\\t15.2023.08.13\\data_train.hdf5', 'trial_key': 'trial_0000', 'trial_num': 0}, {'global_id': 1, 'file_path': 'c:\\code\\brain2text_bundle\\brain2text_project\\data\\t15.2023.08.13\\data_train.hdf5', 'trial_key': 'trial_0001', 'trial_num': 1}]
[{'day_idx': 0, 'global_id': 0, 'file_path': 'c:\\code\\brain2text_bundle\\brain2text_project\\data\\t15.2023.08.13\\data_val.hdf5', 'trial_key': 'trial_0000', 'trial_num': 0}, {'day_idx': 0, 'global_id': 1, 'file_path': 'c:\\code\\brain2text_bundle\\brain2text_project\\data\\t15.2023.08.13\\data_val.hdf5', 'trial_key': 'trial_0001', 'trial_num': 1}]


**Ce qu'on obtient** : deux listes (`train_index`, `val_index`) de dictionnaires.
Chaque dictionnaire pointe vers un trial precis (fichier HDF5 + cle `trial_###`).
**Pourquoi c'est utile** : ces listes servent de 'catalogue' pour charger les essais un par un dans le DataLoader maison.

## Etape 5 ? Charger un trial (features + labels)
On charge un essai individuel depuis un HDF5 et on obtient les tenseurs `x` (features) et `y` (labels).

In [17]:
def load_trial(item: dict):
    file_path = item["file_path"]
    trial_key = item["trial_key"]
    day_idx = item.get("day_idx", 0)
    with h5py.File(file_path, "r") as f:
        g = f[trial_key]
        x_np = g[NEURAL_KEY][:]
        y_np = g[LABELS_KEY][:]
    # remet en [T, 512] si besoin (comme dans le script)
    x_np = x_np.T if x_np.shape[0] == N_FEATURES else x_np
    x = torch.from_numpy(x_np).float()
    y = torch.from_numpy(y_np).long()
    return {"trial_num": item["trial_num"], "x": x, "y": y, "day_idx": day_idx}

# Exemple: charger le 1er trial du train
if len(train_index) > 0:
    sample = load_trial(train_index[0])
    print("trial_num:", sample["trial_num"])
    print("x shape:", sample["x"].shape)
    print("y shape:", sample["y"].shape)


trial_num: 0
x shape: torch.Size([1023, 512])
y shape: torch.Size([500])


**Ce que sont `x` et `y`**
- `x` : les features neurales du trial, forme `[T, 512]` (T = longueur temporelle).
- `y` : la sequence de labels `seq_class_ids`, forme `[L]` (ici L=500 avec padding 0 a droite).
  Les vrais labels sont `y[y != 0]`.

## Etape 6 ? Construire un batch (padding + concat labels)
On assemble plusieurs trials en un batch: `x_batch` padde, `targets` concatene, et longueurs `input_lengths` / `target_lengths`.

On regroupe plusieurs trials pour creer un batch: on padde les sequences `x` jusqu'au meme T_max,
on concatene tous les labels non nuls dans `targets`, et on garde les longueurs `input_lengths` et `target_lengths`.
**A quoi ca sert** : la CTC a besoin des longueurs reelles pour savoir quelles parties du batch sont du padding.
**Exemple de forme** : `x_batch` -> `[B, T_max, 512]`, `targets` -> `[sum_L]`, `input_lengths` -> `[B]`, `target_lengths` -> `[B]`.

In [18]:
def make_batch_from_items(items, device):
    """
    items: liste de dicts {"trial_num": int, "x": Tensor[T_i,512], "y": Tensor[L_i]}
    Retourne:
      - x_batch: [B, T_max, 512]
      - targets: concatenation des labels (sans padding)
      - input_lengths: [B] (T_i)
      - target_lengths: [B] (len labels non-nuls)
    """
    B = len(items)
    assert B > 0, "make_batch_from_items appele avec B=0"

    lengths = [it["x"].shape[0] for it in items]
    T_max = max(lengths)

    x_batch = torch.zeros(B, T_max, N_FEATURES, dtype=torch.float32)
    targets_list = []
    target_lengths = []
    day_indices = []

    for b, it in enumerate(items):
        x = it["x"]
        T_i = x.shape[0]
        x_batch[b, :T_i] = x
        y = it["y"]
        y_nopad = y[y != 0]
        targets_list.append(y_nopad)
        target_lengths.append(y_nopad.numel())
        day_indices.append(it.get('day_idx', 0))

    targets = torch.cat(targets_list, dim=0)

    x_batch = x_batch.to(device, non_blocking=True)
    targets = targets.to(device, non_blocking=True)
    input_lengths = torch.tensor(lengths, dtype=torch.int64, device=device)
    target_lengths = torch.tensor(target_lengths, dtype=torch.int64, device=device)
    day_indices = torch.tensor(day_indices, dtype=torch.int64, device=device)

    return x_batch, targets, input_lengths, target_lengths, day_indices

# Exemple: batch de 2 trials
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

items = [load_trial(train_index[0]), load_trial(train_index[1])]
x_batch, targets, il, tl, day_idx = make_batch_from_items(items, device)
print("x_batch shape:", x_batch.shape)
print("targets shape:", targets.shape)
print("input_lengths:", il.tolist())
print("target_lengths:", tl.tolist())


x_batch shape: torch.Size([2, 1210, 512])
targets shape: torch.Size([88])
input_lengths: [1023, 1210]
target_lengths: [42, 46]


## Etape 7 ? Data augmentation sur un batch
On applique les augmentations directement sur `x_batch` avant le forward du modele.

In [19]:
def gauss_smooth(inputs, device, smooth_kernel_std=2.0, smooth_kernel_size=100, padding="same"):
    """Lissage gaussien 1D le long du temps. inputs: [B,T,C]."""
    assert inputs.dim() == 3, "gauss_smooth attend un tenseur [B,T,C]"
    _, _, C = inputs.shape

    impulse = np.zeros(smooth_kernel_size, dtype=np.float32)
    impulse[smooth_kernel_size // 2] = 1.0
    g = gaussian_filter1d(impulse, smooth_kernel_std)
    keep = np.argwhere(g > 0.01)
    g = np.squeeze(g[keep])
    g = (g / np.sum(g)).astype(np.float32)
    K = int(g.shape[0])

    kern = torch.from_numpy(g).to(device=device, dtype=torch.float32).view(1, 1, K)
    x = inputs.permute(0, 2, 1).contiguous()  # [B,C,T]
    kern = kern.repeat(C, 1, 1)

    if padding == "same":
        pad_left = K // 2
        pad_right = K - 1 - pad_left
        x = F.pad(x, (pad_left, pad_right), mode="reflect")
        y = F.conv1d(x, kern, padding=0, groups=C)
    elif padding == "valid":
        y = F.conv1d(x, kern, padding=0, groups=C)
    else:
        try:
            y = F.conv1d(x, kern, padding=padding, groups=C)
        except Exception:
            pad_left = K // 2
            pad_right = K - 1 - pad_left
            x = F.pad(x, (pad_left, pad_right), mode="reflect")
            y = F.conv1d(x, kern, padding=0, groups=C)

    return y.permute(0, 2, 1).contiguous()  # [B,T,C]

# Exemple d'aug sur un batch
if 'x_batch' in globals():
    x_aug = x_batch
    if AUG_SMOOTH_ENABLE and (np.random.rand() < AUG_SMOOTH_PROB):
        x_aug = gauss_smooth(x_aug, device=device, smooth_kernel_std=AUG_SMOOTH_STD,
                             smooth_kernel_size=AUG_SMOOTH_SIZE, padding=AUG_SMOOTH_PADDING)
    if np.random.rand() < 0.5:
        x_aug = x_aug + torch.randn_like(x_aug) * NOISE_STD
    if np.random.rand() < 0.5:
        B_aug, T_aug, C_aug = x_aug.shape
        offset = torch.randn(B_aug, 1, C_aug, device=x_aug.device) * OFFSET_STD
        x_aug = x_aug + offset
    if RANDOM_CUT_MAX > 0:
        cut = np.random.randint(0, RANDOM_CUT_MAX + 1)
        if cut > 0:
            x_aug = x_aug[:, cut:, :]
    print('x_aug shape:', x_aug.shape)


x_aug shape: torch.Size([2, 1207, 512])


**Toy example (avant / apres lissage)**
On part d'un petit vecteur [0, 1, 0, 0, 1, 0] et on observe l'effet du lissage gaussien.

In [20]:
toy = torch.tensor([0, 1, 0, 0, 1, 0], dtype=torch.float32)
toy = toy.view(1, -1, 1)  # [B=1, T=6, C=1]
toy_sm = gauss_smooth(toy, device=toy.device, smooth_kernel_std=1.0, smooth_kernel_size=7, padding="same")
print('avant :', toy.view(-1).tolist())
print('apres :', [round(v, 3) for v in toy_sm.view(-1).tolist()])


avant : [0.0, 1.0, 0.0, 0.0, 1.0, 0.0]
apres : [0.488, 0.457, 0.299, 0.299, 0.457, 0.488]


## Etape 8 ? Definir le modele (PositionalEncoding + TransformerCTC)
On instancie l'encodeur Transformer et sa positional encoding, puis la tete de sortie CTC.

In [21]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 20000):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)) # C’est la partie qui permet d’avoir des sinusoïdes avec différentes fréquences
        pe[:, 0::2] = torch.sin(pos * div)  # Remplit les colonnes paires (0,2,4,…) avec sin(position * div).
        pe[:, 1::2] = torch.cos(pos * div)  # Remplit les colonnes impaires (1,3,5,…) avec cos(position * div)
        self.register_buffer("pe", pe, persistent=False)

    def forward(self, x):  # [B,T,D]
        T = x.size(1)
        x = x + self.pe[:T].unsqueeze(0)
        return self.dropout(x)




**Explication du Positional Encoding**
Le Transformer n'a pas de notion d'ordre temporel par defaut.
Ici on ajoute un vecteur sinusoidal (sin/cos) a chaque pas de temps pour encoder la position.
Cela permet au modele de distinguer le debut/la fin et d'apprendre des dependances temporelles.

In [22]:
class DayInputLayer(nn.Module):
    """
    Projection specifique par jour :
    x -> W_day x + b_day, puis Softsign + Dropout
    """
    def __init__(self, neural_dim: int, n_days: int, input_dropout: float = 0.2):
        super().__init__()
        self.neural_dim = neural_dim
        self.n_days = n_days
        self.day_activation = nn.Softsign()
        self.day_dropout = nn.Dropout(input_dropout)
        # Matrices W_d (une par jour), initialisees a l'identite
        day_weights = []
        for _ in range(self.n_days):
            weight = nn.Parameter(torch.eye(self.neural_dim))
            day_weights.append(weight)
        self.day_weights = nn.ParameterList(day_weights)
        
        # Biais b_d (un vecteur par jour), initialises a 0
        day_biases = []
        for _ in range(self.n_days):
            bias = nn.Parameter(torch.zeros(1, self.neural_dim))
            day_biases.append(bias)
        self.day_biases = nn.ParameterList(day_biases)

    def forward(self, x: torch.Tensor, day_idx: torch.Tensor) -> torch.Tensor:
        day_weights = torch.stack([self.day_weights[i] for i in day_idx], dim=0)  # [B,D,D]
        day_biases = torch.cat([self.day_biases[i] for i in day_idx], dim=0).unsqueeze(1)  # [B,1,D]
        x = torch.einsum('btd,bdk->btk', x, day_weights) + day_biases
        x = self.day_activation(x)
        x = self.day_dropout(x)
        return x


In [23]:
class TransformerCTC(nn.Module):
    def __init__(self, n_features=512, n_classes=41,
                 d_model=256, nhead=8, num_layers=6,
                 dim_ff=1024, dropout=0.1, blank_id=0,
                 n_days=1, input_dropout=0.2):
        super().__init__()
        self.blank_id = int(blank_id)
        self.in_norm  = nn.LayerNorm(n_features)
        self.proj_in  = nn.Linear(n_features, d_model)
        self.day_input = DayInputLayer(neural_dim=n_features, n_days=n_days, input_dropout=input_dropout)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=dim_ff, dropout=dropout,
            batch_first=True, activation="gelu"
        )
        self.encoder  = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.posenc   = PositionalEncoding(d_model, dropout=dropout)
        self.head     = nn.Linear(d_model, n_classes)
        with torch.no_grad():
            if self.head.bias is not None:
                self.head.bias.zero_()

    def forward(self, x, day_idx):
        x = self.day_input(x, day_idx)  # day-specific projection
        x = self.in_norm(x)             # normalise les features par dimension
        x = self.proj_in(x)             # projette vers l'espace d_model
        x = self.posenc(x)              # ajoute l'information de position
        y = self.encoder(x)             # encodeur Transformer (self-attention)
        logits = self.head(y)           # projection finale vers les classes
        return logits                  # sorties CTC [B,T,C]


In [ ]:
# --- Patching temporel (avant l'encodeur) ---
if 'PATCH_SIZE' not in globals():
    PATCH_SIZE = 14
if 'PATCH_STRIDE' not in globals():
    PATCH_STRIDE = 4

def patch_input_lengths(il, patch_size=PATCH_SIZE, patch_stride=PATCH_STRIDE):
    if patch_size <= 0:
        return il
    if not torch.is_tensor(il):
        il = torch.tensor(il, dtype=torch.int64)
    out = il.clone()
    mask = out >= patch_size
    out[mask] = ((out[mask] - patch_size) // patch_stride) + 1
    return out

class TransformerCTC(nn.Module):
    def __init__(self, n_features=512, n_classes=41,
                 d_model=256, nhead=8, num_layers=6,
                 dim_ff=1024, dropout=0.1, blank_id=0,
                 n_days=1, input_dropout=0.2,
                 patch_size=PATCH_SIZE, patch_stride=PATCH_STRIDE):
        super().__init__()
        self.blank_id = int(blank_id)
        self.patch_size = int(patch_size)
        self.patch_stride = int(patch_stride)
        self.day_input = DayInputLayer(neural_dim=n_features, n_days=n_days, input_dropout=input_dropout)

        if self.patch_size > 0:
            in_dim = n_features * self.patch_size
        else:
            in_dim = n_features

        self.in_norm  = nn.LayerNorm(in_dim)
        self.proj_in  = nn.Linear(in_dim, d_model)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=dim_ff, dropout=dropout,
            batch_first=True, activation='gelu'
        )
        self.encoder  = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.posenc   = PositionalEncoding(d_model, dropout=dropout)
        self.head     = nn.Linear(d_model, n_classes)
        with torch.no_grad():
            if self.head.bias is not None:
                self.head.bias.zero_()

    def _patch(self, x):
        if self.patch_size <= 0:
            return x
        B, T, C = x.shape
        if T < self.patch_size:
            return x
        x_u = x.unsqueeze(1)
        x_u = x_u.permute(0, 3, 1, 2)
        x_u = x_u.unfold(3, self.patch_size, self.patch_stride)
        x_u = x_u.squeeze(2)
        x_u = x_u.permute(0, 2, 3, 1)
        x_p = x_u.reshape(B, x_u.size(1), self.patch_size * C)
        return x_p

    def forward(self, x, day_idx):
        x = self.day_input(x, day_idx)
        x = self._patch(x)
        x = self.in_norm(x)
        x = self.proj_in(x)
        x = self.posenc(x)
        y = self.encoder(x)
        logits = self.head(y)
        return logits


## Etape 9 ? Forward du batch (logits)
On passe `x_batch` dans le modele pour obtenir les logits [B, T, C].

In [24]:
if 'x_batch' in globals():
    if 'DAY_INPUT_DROPOUT' not in globals():
        DAY_INPUT_DROPOUT = 0.2

    model = TransformerCTC(
        n_features=N_FEATURES,
        n_classes=N_CLASSES,
        d_model=TF_D_MODEL,
        nhead=TF_NHEAD,
        num_layers=TF_LAYERS,
        dim_ff=TF_FF_DIM,
        dropout=TF_DROPOUT,
        blank_id=BLANK_ID,
        n_days=N_DAYS,
        input_dropout=DAY_INPUT_DROPOUT
    )

    model = model.to(device)
    logits = model(x_batch, day_idx)
    print('logits shape:', logits.shape)


logits shape: torch.Size([2, 1210, 41])


## Etape 10 ? Calcul de la loss CTC
On applique log-softmax et la CTCLoss avec les longueurs reelles.

In [25]:
ctc = nn.CTCLoss(blank=BLANK_ID, zero_infinity=True, reduction='mean')
if 'logits' in globals():
    logp_TBC = F.log_softmax(logits.transpose(0, 1), dim=-1)  # [T,B,C]
    il_ctc = patch_input_lengths(il)
    il_ctc = torch.clamp(il_ctc, max=logp_TBC.size(0))
    loss_ctc = ctc(logp_TBC, targets, il_ctc, tl)
    print('loss_ctc:', float(loss_ctc))


loss_ctc: 73.76786804199219


## Etape 11 ? Backward + update (optimizer)
On calcule les gradients, clippe si besoin, puis on met a jour les poids.

In [26]:
# Optim + scheduler (identiques au script)
# Assure que le modele existe avant l'optimiseur
if 'model' not in globals():
    model = TransformerCTC(
        n_features=N_FEATURES,
        n_classes=N_CLASSES,
        d_model=TF_D_MODEL,
        nhead=TF_NHEAD,
        num_layers=TF_LAYERS,
        dim_ff=TF_FF_DIM,
        dropout=TF_DROPOUT,
        blank_id=BLANK_ID,
        n_days=N_DAYS,
        input_dropout=DAY_INPUT_DROPOUT
    )
if 'device' in globals():
    model = model.to(device)
# Optimiseur AdamW avec weight decay
optim = torch.optim.AdamW(model.parameters(), lr=BASE_LR, weight_decay=WEIGHT_DECAY)  
#  Nombre de batches par epoch (taille dataset / batch)
num_batches_per_epoch = max(1, math.ceil(len(train_index) / BATCH_SIZE))  
# Nombre de steps d'optim par epoch 
steps_per_epoch = max(1, math.ceil(num_batches_per_epoch / ACCUM_STEPS))
# Total des steps sur tout l'entrainement
total_steps = max(1, EPOCHS * steps_per_epoch)
# Steps de warmup (au debut LR augmente progressivement)
warmup_steps = max(1, int(WARMUP_FRAC * total_steps))  # fraction de warmup

# Fonction de schedule: warmup lineaire puis cosine decay
def lr_lambda(step: int):
    # Warmup: LR monte de 0 -> 1 sur warmup_steps
    if step < warmup_steps:
        return float(step) / float(max(1, warmup_steps))
    # Cosine decay: 1 -> 0 sur le reste des steps
    progress = float(step - warmup_steps) / float(max(1, total_steps - warmup_steps))
    # Clamp pour eviter les depassements numeriques
    progress = min(max(progress, 0.0), 1.0)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

# 7) Scheduler PyTorch base sur la fonction ci-dessus
scheduler = torch.optim.lr_scheduler.LambdaLR(optim, lr_lambda=lr_lambda)

# Un seul step d'optimisation pour l'exemple
if 'loss_ctc' in globals():
    optim.zero_grad(set_to_none=True)
    loss_ctc.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_NORM)
    optim.step()
    scheduler.step()
    print('step done, lr:', optim.param_groups[0]['lr'])


step done, lr: 1.5151515151515152e-06


## Etape 12 ? Validation (greedy PER) + sauvegarde
On decode en greedy CTC, calcule le PER, et on peut sauvegarder les meilleurs poids.

In [27]:
# Decode CTC logits with greedy argmax, collapse repeats, and drop blanks.
@torch.no_grad()
def ctc_greedy_ids_from_logits(logits, blank_id=0):
    if logits.dim() == 3:
        logits = logits.squeeze(0)  # [T,C]
    ids = torch.argmax(logits, dim=-1).tolist()
    out, prev = [], None
    for i in ids:
        if i == blank_id:
            prev = i
            continue
        if prev is None or i != prev:
            out.append(int(i))
        prev = i
    return out

# Compute Levenshtein edit distance between two sequences.
def levenshtein(a, b):
    dp = list(range(len(b) + 1))
    for i, x in enumerate(a, start=1):
        prev = dp[0]
        dp[0] = i
        for j, y in enumerate(b, start=1):
            cur = dp[j]
            dp[j] = prev if x == y else 1 + min(prev, dp[j], dp[j - 1])
            prev = cur
    return dp[-1]

# Compute phoneme error rate (PER) using Levenshtein distance.
def phoneme_error_rate(pred, ref, blank_id=0):
    r = [int(i) for i in ref if int(i) != 0]
    if not r:
        return 0.0 if not pred else float('nan')
    return levenshtein(pred, r) / len(r)

@torch.no_grad()
def eval_on_validation(model, val_index, device, blank_id=0):
    model.eval()
    pers = []
    sample = None
    for item in val_index[:3]:  # mini-eval sur 3 trials pour l'exemple
        if sample is None:
            sample = item
        batch = load_trial(item)
        x = batch['x'].unsqueeze(0).to(device)
        y = batch['y'].to(device)
        targets = y[y != 0].tolist()
        if not targets:
            continue
        day_idx = torch.tensor([batch.get('day_idx', 0)], dtype=torch.int64, device=device)
        logits = model(x, day_idx)
        pred_ids = ctc_greedy_ids_from_logits(logits, blank_id=blank_id)
        if sample is item:
            print('REF :', ' '.join(map(str, targets)))
            print('PRED:', ' '.join(map(str, pred_ids)))
        per = phoneme_error_rate(pred_ids, targets, blank_id=blank_id)
        if per == per:
            pers.append(per)
    model.train()
    return float(sum(pers) / len(pers)) if pers else float('nan')

if len(val_index) > 0:
    per_g = eval_on_validation(model, val_index, device, blank_id=BLANK_ID)
    print('PER (greedy, mini-eval):', per_g)


REF : 37 34 40 20 2 23 40 29 18 40 10 3 40 20 25 9 40 2 31 40 10 17 29 40 27 26 23 31 40 2 38 40 36 11 21 40
PRED: 32 26 32 28 32 17 32 17 32 17 32 17 32 17 32 12 17 32 17 32 28 17 32 17 32 32 29 32 29 32 26 32 32 29 32 17 32 17 32 9 32 29 32 29 32 28 32 26 32 28 32 29 17 32 26 28 17 32 17 29 17 29 32 17 32 17 32 17 26 17 32 29 17 32 17 29 17 29 17 29 20 32 32 26 29 17 32 26 17 32 17 32 26 32 17 29 17 32 17 32 17 32 29 17 29 17 17 29 29 32 29 32 26 29 32 29 26 17 12 29 32 26 29 32 17 32 29 32 29 28 29 17 32 26 29 26 17 26 29 17 26 17 26 17 26 17 29 26 17 32 17 32 26 29 32 17 29 17 12 29 17 29 17 29 17 29 17 29 17 12 32 17 32 17 32 17 32 17 26 17 29 17 26 17 26 17 32 17 32 29 32 12 17 32 29 32 17 32 29 17 32 29 32 17 32 17 32 26 17 32 29 17 26 17 32 26 17 32 17 26 26 17 32 17 32 26 17 26 32 26 32 29 32 26 32 29 17 32 17 29 32 29 12 32 12 29 17 25 29 17 29 17 29 32 17 12 32 29 32 17 32 17 26 17 26 29 17 26 32 26 29 26 29 32 26 17 32 26 29 26 29 12 29 32 29 32 12 32 12 32 12 26 29 26 29 3

## Etape 13 ? Main: boucle d'entra?nement complete
On assemble tout: index -> batch -> data-aug -> forward -> loss -> backward -> eval.

In [ ]:
def main_train():
    # device + seed
    set_seed(SEED)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if device.type == 'cuda':
        torch.set_float32_matmul_precision('high')
        cudnn.benchmark = True
        cudnn.deterministic = False

    # index
    train_index = build_train_index_from_hdf5()
    val_index = globals().get('val_index')
    if val_index is None:
        val_index = build_eval_index_from_hdf5()

    # run dir + config
    run_id = datetime.now().strftime('%Y%m%d-%H%M%S')
    run_dir = Path(RUNS_ROOT) / run_id
    run_dir.mkdir(parents=True, exist_ok=False)

    config = {
        'run_id': run_id,
        'data_root': DATA_ROOT,
        'split_csv': SPLIT_CSV,
        'n_days': N_DAYS,
        'sessions': SESSIONS,
        'model': {
            'n_features': N_FEATURES,
            'n_classes': N_CLASSES,
            'blank_id': BLANK_ID,
            'd_model': TF_D_MODEL,
            'nhead': TF_NHEAD,
            'num_layers': TF_LAYERS,
            'dim_ff': TF_FF_DIM,
            'dropout': TF_DROPOUT,
            'day_input_dropout': DAY_INPUT_DROPOUT,
            'patch_size': PATCH_SIZE,
            'patch_stride': PATCH_STRIDE,
        },
        'train': {
            'base_lr': BASE_LR,
            'weight_decay': WEIGHT_DECAY,
            'clip_norm': CLIP_NORM,
            'epochs': EPOCHS,
            'batch_size': BATCH_SIZE,
            'accum_steps': ACCUM_STEPS,
            'eval_every': EVAL_EVERY,
            'save_last_every': SAVE_LAST_EVERY,
            'use_amp': USE_AMP,
        },
        'aug': {
            'smooth_enable': AUG_SMOOTH_ENABLE,
            'smooth_prob': AUG_SMOOTH_PROB,
            'smooth_std': AUG_SMOOTH_STD,
            'smooth_size': AUG_SMOOTH_SIZE,
            'smooth_padding': AUG_SMOOTH_PADDING,
            'noise_std': NOISE_STD,
            'offset_std': OFFSET_STD,
            'random_cut_max': RANDOM_CUT_MAX,
        },
    }
    with open(run_dir / 'config.json', 'w', encoding='utf-8') as f:
        json.dump(config, f, ensure_ascii=False, indent=2)
    print(f'[SAVE] Run dir: {run_dir}')

    # model
    model = TransformerCTC(
        n_features=N_FEATURES,
        n_classes=N_CLASSES,
        d_model=TF_D_MODEL,
        nhead=TF_NHEAD,
        num_layers=TF_LAYERS,
        dim_ff=TF_FF_DIM,
        dropout=TF_DROPOUT,
        blank_id=BLANK_ID,
        n_days=N_DAYS,
        input_dropout=DAY_INPUT_DROPOUT
    ).to(device)

    ctc = nn.CTCLoss(blank=BLANK_ID, zero_infinity=True, reduction='mean')
    optim = torch.optim.AdamW(model.parameters(), lr=BASE_LR, weight_decay=WEIGHT_DECAY)

    num_batches_per_epoch = max(1, math.ceil(len(train_index) / BATCH_SIZE))
    steps_per_epoch = max(1, math.ceil(num_batches_per_epoch / ACCUM_STEPS))
    total_steps = max(1, EPOCHS * steps_per_epoch)
    warmup_steps = max(1, int(WARMUP_FRAC * total_steps))

    def lr_lambda(step: int):
        if step < warmup_steps:
            return float(step) / float(max(1, warmup_steps))
        progress = float(step - warmup_steps) / float(max(1, total_steps - warmup_steps))
        progress = min(max(progress, 0.0), 1.0)
        return 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optim, lr_lambda=lr_lambda)

    best_per = float('inf')

    # train loop
    for epoch in range(1, EPOCHS + 1):
        model.train()
        order = np.random.permutation(len(train_index))
        num_batches = int(np.ceil(len(order) / BATCH_SIZE))
        optim.zero_grad(set_to_none=True)
        accum_counter = 0
        losses_epoch = []

        for b_idx in range(num_batches):
            start = b_idx * BATCH_SIZE
            end = min((b_idx + 1) * BATCH_SIZE, len(order))
            batch_ids = order[start:end]

            items = [load_trial(train_index[j]) for j in batch_ids]
            if len(items) == 0:
                continue

            x_batch, targets, il, tl, day_idx = make_batch_from_items(items, device)
            if x_batch.shape[0] == 0:
                continue

            accum_counter += 1

            # data aug
            if AUG_SMOOTH_ENABLE and (np.random.rand() < AUG_SMOOTH_PROB):
                x_batch = gauss_smooth(x_batch, device=device, smooth_kernel_std=AUG_SMOOTH_STD,
                                      smooth_kernel_size=AUG_SMOOTH_SIZE, padding=AUG_SMOOTH_PADDING)
            if np.random.rand() < 0.5:
                x_batch = x_batch + torch.randn_like(x_batch) * NOISE_STD
            if np.random.rand() < 0.5:
                B_aug, T_aug, C_aug = x_batch.shape
                offset = torch.randn(B_aug, 1, C_aug, device=x_batch.device) * OFFSET_STD
                x_batch = x_batch + offset
            if RANDOM_CUT_MAX > 0:
                cut = np.random.randint(0, RANDOM_CUT_MAX + 1)
                if cut > 0:
                    x_batch = x_batch[:, cut:, :]
                    il = il - cut

            il = patch_input_lengths(il)

            logits = model(x_batch, day_idx)
            logp_TBC = F.log_softmax(logits.transpose(0, 1), dim=-1)
            loss_ctc = ctc(logp_TBC, targets, il, tl)
            loss = loss_ctc / ACCUM_STEPS
            loss.backward()

            if (accum_counter % ACCUM_STEPS) == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_NORM)
                optim.step()
                optim.zero_grad(set_to_none=True)
                scheduler.step()

            losses_epoch.append(float(loss_ctc.item()))
            if (b_idx % PRINT_EVERY) == 0:
                print(f'[epoch {epoch:02d}] batch {b_idx:06d} | loss_ctc {loss_ctc.item():.4f}')

        # flush if gradients left
        if accum_counter > 0 and (accum_counter % ACCUM_STEPS) != 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_NORM)
            optim.step()
            optim.zero_grad(set_to_none=True)
            scheduler.step()

        # save last
        if (epoch % SAVE_LAST_EVERY) == 0 or epoch == EPOCHS:
            torch.save({
                'model': model.state_dict(),
                'epoch': epoch,
            }, run_dir / 'model_last.pt')
            print(f'[SAVE] model_last.pt (epoch {epoch})')

        if (epoch % EVAL_EVERY) == 0:
            per_g = eval_on_validation(model, val_index, device, blank_id=BLANK_ID)
            print(f'[Epoch {epoch:02d} | EVAL] PER_g={per_g:.3f}')
            if per_g == per_g and per_g < best_per:
                best_per = per_g
                torch.save({
                    'model': model.state_dict(),
                    'epoch': epoch,
                    'per_g': float(per_g),
                }, run_dir / 'model_bestPER.pt')
                print(f'[SAVE] model_bestPER.pt (PER={per_g:.4f})')

    return model

trained_model = main_train()


[TRAIN] Trials total indexes: 1680
[epoch 01] batch 000000 | loss_ctc 85.9489
[epoch 01] batch 000050 | loss_ctc 53.7794
[epoch 01] batch 000100 | loss_ctc 10.7404
